In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Dataset path:", path)


print(os.listdir(path))

Using Colab cache for faster access to the 'lending-club' dataset.
Dataset path: /kaggle/input/lending-club
['rejected_2007_to_2018Q4.csv.gz', 'accepted_2007_to_2018Q4.csv.gz', 'accepted_2007_to_2018q4.csv', 'rejected_2007_to_2018q4.csv']


In [2]:
import pandas as pd
import os
# Download dataset
path = kagglehub.dataset_download("wordsforthewise/lending-club")

df = pd.read_csv(
    os.path.join(path, "accepted_2007_to_2018Q4.csv.gz"),
    low_memory=False
)

Using Colab cache for faster access to the 'lending-club' dataset.


DATA EXPLORATION

In [3]:
df.shape
# total rows and columns

df.columns.tolist()
# list all column names


['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [4]:
df.info(verbose=True, show_counts=True)
# dtypes and non-null counts per column

df.describe()
# summary stats for numeric columns

df.describe(include='object')
# summary stats for categorical/text columns



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 151 columns):
 #    Column                                      Non-Null Count    Dtype  
---   ------                                      --------------    -----  
 0    id                                          2260701 non-null  object 
 1    member_id                                   0 non-null        float64
 2    loan_amnt                                   2260668 non-null  float64
 3    funded_amnt                                 2260668 non-null  float64
 4    funded_amnt_inv                             2260668 non-null  float64
 5    term                                        2260668 non-null  object 
 6    int_rate                                    2260668 non-null  float64
 7    installment                                 2260668 non-null  float64
 8    grade                                       2260668 non-null  object 
 9    sub_grade                                   

,id,term,grade,sub_grade,emp_title,emp_length,home_ownership,verification_status,issue_d,loan_status,...,hardship_status,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_loan_status,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date
count,2260701,2260668,2260668,2260668,2093699,2113761,2260668,2260668,2260668,2260668,...,10917,10917,10917,10917,10917,2260668,2260668,34246,34246,34246
unique,2260701,2,7,35,512694,11,6,3,139,9,...,3,27,28,27,5,2,2,83,3,90
top,Total amount funded in policy code 2: 521953170,36 months,B,C1,Teacher,10+ years,MORTGAGE,Source Verified,Mar-2016,Fully Paid,...,COMPLETED,Sep-2017,Dec-2017,Sep-2017,Late (16-30 days),Cash,N,Feb-2019,ACTIVE,Jan-2019
freq,1,1609754,663557,145903,38824,748005,1111450,886231,61992,1076751,...,7819,2444,1756,1715,4770,2182546,2226422,2606,14704,1710


In [5]:
df.isnull().sum().sort_values(ascending=False)
# count of missing values per column, sorted

,0
member_id,2260701
orig_projected_additional_accrued_interest,2252050
hardship_reason,2249784
hardship_payoff_balance_amount,2249784
hardship_last_payment_amount,2249784
...,...
total_rec_int,33
disbursement_method,33
hardship_flag,33
debt_settlement_flag,33


In [6]:
(df.isnull().mean()*100).sort_values(ascending=False)
# percentage of missing values per column


,0
member_id,100.000000
orig_projected_additional_accrued_interest,99.617331
hardship_reason,99.517097
hardship_payoff_balance_amount,99.517097
hardship_last_payment_amount,99.517097
...,...
total_rec_int,0.001460
disbursement_method,0.001460
hardship_flag,0.001460
debt_settlement_flag,0.001460


In [7]:
df.nunique().sort_values()
# unique value count per column

,0
member_id,0
policy_code,1
deferral_term,1
hardship_type,1
hardship_length,1
...,...
last_pymnt_amnt,704467
total_pymnt_inv,1311099
total_pymnt,1633864
url,2260668


In [8]:
df['loan_status'].value_counts()
# check target variable distribution

,count
loan_status,
Fully Paid,1076751
Current,878317
Charged Off,268559
Late (31-120 days),21467
In Grace Period,8436
Late (16-30 days),4349
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761
Default,40


DATA CLEANING

Good customer:

- Fully Paid → 1,076,751 records

Bad customer:

We will combine the following categories into a single “bad customer” class:

- Charged Off :268,559 records
- Default : 40 records
- Late (31–120 days) : 21,467 records

The following category will be removed from the analysis:

- Late (16–30 days) : 4,349 records

In [9]:
# clean loan_status column
# remove rows that don't fit either class


remove_status = ['Current', 'Late (16-30 days)', 'In Grace Period',
                  'Does not meet the credit policy. Status:Fully Paid',
                  'Does not meet the credit policy. Status:Charged Off']

df = df[~df['loan_status'].isin(remove_status)]

# map to binary target
bad_status = ['Charged Off', 'Default', 'Late (31-120 days)']

df['loan_status'] = df['loan_status'].apply(lambda x: 1 if x in bad_status else 0)

df['loan_status'].value_counts()



,count
loan_status,
0,1076784
1,290066


DATA LEAKAGE

In [10]:
to_drop = [
    # Post-approval funding
    'funded_amnt', 'funded_amnt_inv',
    # LC internal scoring
    'int_rate', 'installment', 'grade', 'sub_grade',
    # High cardinality / free text
    'emp_title', 'title', 'zip_code', 'url', 'desc',
    # Post-origination payments
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
    'last_pymnt_amnt', 'next_pymnt_d', 'pymnt_plan',
    # Post-origination credit
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    # Hardship
    'hardship_flag', 'hardship_reason', 'hardship_status', 'hardship_amount',
    'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date',
    'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    # Settlement
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage',
    'settlement_term',
]

to_drop = [col for col in to_drop if col in df.columns]
df = df.drop(columns=to_drop)

print(f"Dropped  : {len(to_drop)} columns")
print(f"Remaining: {len(df.columns)} columns")

Dropped  : 46 columns
Remaining: 105 columns


# POST-APPROVAL FUNDING
# funded_amnt: The amount committed by investors - only known after approval decision, not at application time
# funded_amnt_inv: The amount funded by investors - post-approval, not available at application

# LC INTERNAL RISK SCORING
# int_rate: Interest rate assigned by LendingClub - set after their internal model runs, not a borrower attribute
# installment: Monthly payment amount - derived from int_rate + term + loan_amnt, post-decision calculation
# grade: LendingClub's internal risk grade (A-G) - assigned by LC's own model, not a raw borrower attribute
# sub_grade: More granular version of LC's grade (A1-G5) - same issue as grade

# HIGH CARDINALITY / FREE TEXT
# emp_title: Job title typed freely by borrower - 500k+ unique variations, too noisy to use directly
# title: Free-text loan title written by borrower - redundant with purpose column which is already standardized
# zip_code: Borrower's zip code - hundreds of unique values, too granular without external enrichment
# url: Link to the LendingClub loan page - administrative metadata, zero predictive value
# desc: Free-text loan description written by borrower - over 80% missing and unstructured

# POST-ORIGINATION PAYMENT BEHAVIOR
# out_prncp: Remaining outstanding principal - only exists and updates during repayment period
# out_prncp_inv: Investor's portion of remaining principal - same issue as out_prncp
# total_pymnt: Total amount received so far from borrower - accumulates after loan is issued
# total_pymnt_inv: Total payments received by investors - post-origination accumulation
# total_rec_prncp: Total principal received to date - only known during/after repayment
# total_rec_int: Total interest received to date - only known during/after repayment
# total_rec_late_fee: Total late fees received - only exists after borrower misses payments
# recoveries: Amount recovered after charge-off - only populated after borrower defaults
# collection_recovery_fee: Fee charged during recovery process - only after default and collections
# last_pymnt_d: Date of the last payment made - only exists after loan is active and payments begin
# last_pymnt_amnt: Amount of the last payment - post-origination, only known after payments start
# next_pymnt_d: Next scheduled payment date - only scheduled after loan is issued and active
# pymnt_plan: Whether borrower is on a payment modification plan - post-origination event

# POST-ORIGINATION CREDIT UPDATES
# last_credit_pull_d: Date LC last pulled borrower's credit - happens after loan is already active
# last_fico_range_high: Most recent FICO high - updated throughout loan life, not the application-time score
# last_fico_range_low: Most recent FICO low - same issue as last_fico_range_high

# HARDSHIP PROGRAM (POST-ORIGINATION EVENT)
# hardship_flag: Indicates if borrower entered a hardship plan - hardship only declared during repayment
# hardship_reason: Reason borrower cited for hardship - post-origination, only exists if hardship declared
# hardship_status: Current status of the hardship plan - post-origination event tracking
# hardship_amount: Amount deferred under hardship plan - post-origination financial adjustment
# hardship_start_date: When the hardship plan began - post-origination date
# hardship_end_date: When the hardship plan ended - post-origination date
# payment_plan_start_date: When the modified payment plan started - post-origination modification
# hardship_dpd: Days past due during hardship period - post-origination repayment behavior
# hardship_loan_status: Loan status during hardship - directly encodes post-origination outcome
# orig_projected_additional_accrued_interest: Projected interest during hardship deferral - post-origination calculation
# hardship_payoff_balance_amount: Loan balance during hardship - post-origination balance
# hardship_last_payment_amount: Last payment made during hardship - post-origination payment behavior

# DEBT SETTLEMENT (POST-DEFAULT EVENT)
# debt_settlement_flag: Indicates borrower entered debt settlement - only flagged after default
# debt_settlement_flag_date: Date settlement flag was set - post-default timestamp
# settlement_status: Current status of the settlement negotiation - post-default process
# settlement_date: Date settlement was reached - post-default event
# settlement_amount: Amount borrower agreed to settle for - post-default negotiation result
# settlement_percentage: Percentage of balance settled - post-default calculation
# settlement_term: Repayment terms of the settlement - post-default agreement


INCONSISTENT VALUES/ INCORRECT DATA TYPES

let's inspect the data

In [11]:
# 1. Full numeric description - no truncation
import pandas as pd
pd.set_option('display.max_rows', 200)

desc = df.select_dtypes(include='float64').describe().T
print(desc.to_string())

                                         count           mean            std     min       25%        50%        75%          max
member_id                                  0.0            NaN            NaN     NaN       NaN        NaN        NaN          NaN
loan_amnt                            1366817.0   14459.653011    8738.443267   500.0   8000.00   12000.00   20000.00     40000.00
annual_inc                           1366817.0   76259.794609   70313.623641     0.0  45760.00   65000.00   90000.00  10999200.00
dti                                  1366419.0      18.311995      11.317034    -1.0     11.80      17.63      24.09       999.00
delinq_2yrs                          1366817.0       0.318443       0.879573     0.0      0.00       0.00       0.00        39.00
fico_range_low                       1366817.0     696.124525      31.816170   625.0    670.00     690.00     710.00       845.00
fico_range_high                      1366817.0     700.124664      31.816822   629.0    67

In [12]:
# 3. Full categorical value counts
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts().to_string())

Streaming output truncated to the last 5000 lines.
90866518                                            1
91008858                                            1
89865943                                            1
91059019                                            1
91198981                                            1
90756340                                            1
91208891                                            1
91068127                                            1
90251526                                            1
91250717                                            1
90765377                                            1
90246870                                            1
91068272                                            1
89937767                                            1
91067785                                            1
90806473                                            1
90996411                                            1
90946928                       

let's fix it

In [13]:



# 1. DATA TYPE FIXES

# term: stored as "36 months" string -> extract number -> convert to integer
df['term'] = df['term'].str.replace(' months', '').astype('Int64')

# emp_length: stored as "10+ years", "< 1 year" etc -> extract number -> convert to float
# "< 1 year" becomes 0, "10+ years" becomes 10
df['emp_length'] = df['emp_length'].replace({
    '< 1 year': '0',
    '10+ years': '10'
})
df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)

# issue_d: stored as "Mar-2016" string -> convert to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')

# earliest_cr_line: stored as "Aug-2001" string -> convert to datetime
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')

# sec_app_earliest_cr_line: same string date issue as earliest_cr_line but for co-borrower
df['sec_app_earliest_cr_line'] = pd.to_datetime(df['sec_app_earliest_cr_line'], format='%b-%Y', errors='coerce')

# initial_list_status: values are lowercase "w" and "f" while all other columns use uppercase
df['initial_list_status'] = df['initial_list_status'].str.upper()

# disbursement_method: "DirectPay" uses CamelCase while "Cash" is plain text -> standardize to uppercase
df['disbursement_method'] = df['disbursement_method'].str.upper()



In [14]:
# 2. IMPOSSIBLE VALUES — UTILIZATION COLUMNS (cannot exceed 100%)

# revol_util: max is 892% which is impossible for a utilization rate
# values above 100 are data errors -> cap at 100, flag extremes as separate category
df['revol_util_flag'] = (df['revol_util'] > 100).astype(int)
df['revol_util'] = df['revol_util'].clip(upper=100)

# il_util: max is 558 which is impossible for installment loan utilization
# same treatment as revol_util
df['il_util_flag'] = (df['il_util'] > 100).astype(int)
df['il_util'] = df['il_util'].clip(upper=100)

# all_util: max is 204% which is impossible for overall utilization
# same treatment
df['all_util_flag'] = (df['all_util'] > 100).astype(int)
df['all_util'] = df['all_util'].clip(upper=100)

# bc_util: max is 339% which is impossible for bank card utilization
# same treatment
df['bc_util_flag'] = (df['bc_util'] > 100).astype(int)
df['bc_util'] = df['bc_util'].clip(upper=100)

# sec_app_revol_util: max is 235% which is impossible for co-borrower revolving utilization
# same treatment
df['sec_app_revol_util_flag'] = (df['sec_app_revol_util'] > 100).astype(int)
df['sec_app_revol_util'] = df['sec_app_revol_util'].clip(upper=100)



In [15]:
 #3. IMPOSSIBLE VALUES — DTI

import numpy as np
# dti: min is -1.0 (impossible, DTI can never be negative)
# and max is 999 (clearly a system placeholder for unknown/error)
# negative values -> set to NaN (genuinely invalid)
# 999 -> set to NaN (placeholder, not a real value)
# flag both cases so the model knows data was missing/corrupt here
# any DTI above a realistic ceiling (e.g. 200) should be treated as error
df['dti_flag'] = ((df['dti'] < 0) | (df['dti'] > 200)).astype(int)
df['dti'] = df['dti'].where((df['dti'] >= 0) & (df['dti'] <= 200), other=np.nan)


In [16]:
#4. PLACEHOLDER VALUES — 9,999,999 SENTINEL VALUES


# total_rev_hi_lim: max is 9,999,999 which is a system placeholder
# meaning "no limit" or "unknown", not a real credit limit
# replace with NaN and flag so model knows these were special cases
df['total_rev_hi_lim_flag'] = (df['total_rev_hi_lim'] == 9999999).astype(int)
df['total_rev_hi_lim'] = df['total_rev_hi_lim'].replace(9999999, np.nan)

# tot_hi_cred_lim: same 9,999,999 placeholder pattern as total_rev_hi_lim
df['tot_hi_cred_lim_flag'] = (df['tot_hi_cred_lim'] == 9999999).astype(int)
df['tot_hi_cred_lim'] = df['tot_hi_cred_lim'].replace(9999999, np.nan)

# mo_sin_old_il_acct: max is 999 which is a placeholder for unknown/missing
# not a real value of 999 months (83 years)
df['mo_sin_old_il_acct_flag'] = (df['mo_sin_old_il_acct'] == 999).astype(int)
df['mo_sin_old_il_acct'] = df['mo_sin_old_il_acct'].replace(999, np.nan)


In [17]:
# 6. RARE CATEGORIES — HOME OWNERSHIP
# home_ownership: ANY (304 rows), OTHER (144 rows), NONE (48 rows)
# are too rare and vague to be meaningful for credit risk modeling
# group all three into a single "OTHER" category
df['home_ownership'] = df['home_ownership'].replace({
    'ANY': 'OTHER',
    'NONE': 'OTHER'
})

In [18]:

# 7. RARE CATEGORIES — ADDR_STATE


# addr_state: Iowa (IA) has only 7 rows out of 1.3 million
# extremely underrepresented and will cause issues with encoding
# flag states with fewer than 100 rows as "OTHER"
state_counts = df['addr_state'].value_counts()
rare_states = state_counts[state_counts < 100].index.tolist()
df['addr_state'] = df['addr_state'].replace(rare_states, 'OTHER')


In [19]:
# VERIFICATION


print(" Data Type Check ")
print(df[['term', 'emp_length', 'issue_d', 'earliest_cr_line']].dtypes)

print("Utilization Columns (should all be <= 100)")
for col in ['revol_util', 'il_util', 'all_util', 'bc_util', 'sec_app_revol_util']:
    if col in df.columns:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}")

print("\ DTI Check ")
print(f"dti: min={df['dti'].min()}, max={df['dti'].max()}, nulls={df['dti'].isna().sum()}")

print(" Placeholder Check")
print(f"total_rev_hi_lim 9999999 remaining: {(df['total_rev_hi_lim'] == 9999999).sum()}")
print(f"tot_hi_cred_lim 9999999 remaining : {(df['tot_hi_cred_lim'] == 9999999).sum()}")

print(" home_ownership categories ")
print(df['home_ownership'].value_counts())

print("\n=== Flag columns created ===")
flag_cols = [col for col in df.columns if col.endswith('_flag')]
print(flag_cols)

print(f"\nFinal shape: {df.shape}")

 Data Type Check 
term                         Int64
emp_length                 float64
issue_d             datetime64[ns]
earliest_cr_line    datetime64[ns]
dtype: object
Utilization Columns (should all be <= 100)
revol_util: min=0.0, max=100.0
il_util: min=0.0, max=100.0
all_util: min=0.0, max=100.0
bc_util: min=0.0, max=100.0
sec_app_revol_util: min=0.0, max=100.0
\ DTI Check 
dti: min=0.0, max=199.72, nulls=650
 Placeholder Check
total_rev_hi_lim 9999999 remaining: 0
tot_hi_cred_lim 9999999 remaining : 0
 home_ownership categories 
home_ownership
MORTGAGE    674904
RENT        543891
OWN         147526
OTHER          496
Name: count, dtype: int64

=== Flag columns created ===
['revol_util_flag', 'il_util_flag', 'all_util_flag', 'bc_util_flag', 'sec_app_revol_util_flag', 'dti_flag', 'total_rev_hi_lim_flag', 'tot_hi_cred_lim_flag', 'mo_sin_old_il_acct_flag']

Final shape: (1366850, 114)


<>:12: SyntaxWarning: invalid escape sequence '\ '
<>:12: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_407/1895637873.py:12: SyntaxWarning: invalid escape sequence '\ '
  print("\ DTI Check ")


let's check again

In [20]:
#  Full categorical value counts
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts().to_string())

Streaming output truncated to the last 5000 lines.
91288411                                            1
89875839                                            1
90865752                                            1
91030472                                            1
91278378                                            1
91288453                                            1
91110405                                            1
91289529                                            1
91000640                                            1
91291484                                            1
91100302                                            1
91210507                                            1
91120751                                            1
91040585                                            1
91070525                                            1
91210534                                            1
91200365                                            1
91230556                       

In [21]:
#  Full numeric description - no truncation
import pandas as pd
pd.set_option('display.max_rows', 200)

desc = df.select_dtypes(include='float64').describe().T
print(desc.to_string())

                                         count           mean            std     min       25%        50%        75%          max
member_id                                  0.0            NaN            NaN     NaN       NaN        NaN        NaN          NaN
loan_amnt                            1366817.0   14459.653011    8738.443267   500.0   8000.00   12000.00   20000.00     40000.00
emp_length                           1286450.0       5.961489       3.692290     0.0      2.00       6.00      10.00        10.00
annual_inc                           1366817.0   76259.794609   70313.623641     0.0  45760.00   65000.00   90000.00  10999200.00
dti                                  1366200.0      18.237979       8.928202     0.0     11.80      17.63      24.08       199.72
delinq_2yrs                          1366817.0       0.318443       0.879573     0.0      0.00       0.00       0.00        39.00
fico_range_low                       1366817.0     696.124525      31.816170   625.0    67

MISSING VALUES HANDLING

In [22]:
# get full picture of missing values with percentage
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing.to_string())

                                     missing_count  missing_pct
member_id                                  1366850       100.00
hardship_length                            1359471        99.46
deferral_term                              1359471        99.46
hardship_type                              1359471        99.46
sec_app_mths_since_last_major_derog        1359374        99.45
sec_app_revol_util                         1346569        98.52
sec_app_open_act_il                        1346202        98.49
sec_app_fico_range_high                    1346202        98.49
sec_app_earliest_cr_line                   1346202        98.49
sec_app_fico_range_low                     1346202        98.49
sec_app_collections_12_mths_ex_med         1346202        98.49
sec_app_inq_last_6mths                     1346202        98.49
revol_bal_joint                            1346203        98.49
sec_app_chargeoff_within_12_mths           1346202        98.49
sec_app_num_rev_accts                   

In [23]:
# check if mths_since_rcnt_il missing pattern matches the 59% credit bureau group
# or if it follows the "never happened" logic

# if it is truly a credit bureau data issue, it should be missing
# for the same rows as the other 59% group columns
same_missing = (
    df['mths_since_rcnt_il'].isna() == df['open_acc_6m'].isna()
).mean()

print(f"Rows where mths_since_rcnt_il and open_acc_6m are BOTH missing: {same_missing:.2%}")

# if this is close to 100%, it belongs to Group 3
# if it is much lower, it belongs to Group 2

Rows where mths_since_rcnt_il and open_acc_6m are BOTH missing: 98.91%


In [24]:
#Drop Ghost Rows
# these are entirely empty rows with no real borrower data behind them
# every real loan must have a loan amount so we drop where loan_amnt is null
before = len(df)
df = df.dropna(subset=['loan_amnt'])

print(f"Ghost rows removed : {before - len(df)}")
print(f"Rows remaining     : {len(df)}")

# verify the 33-missing columns are now clean
print(f"\nMissing in loan_amnt after drop: {df['loan_amnt'].isna().sum()}")
print(f"Missing in term after drop     : {df['term'].isna().sum()}")

Ghost rows removed : 33
Rows remaining     : 1366817

Missing in loan_amnt after drop: 0
Missing in term after drop     : 0


In [25]:
#Goup1 Joint Loan Columns (MAR)
# these columns are missing because the loan is individual, not joint
# missing does not mean unknown - it means there is no second borrower
# we can perfectly verify this using application_type

# first create a clean flag to identify joint applications
# this is useful for the model to know if a loan is joint or not
df['is_joint_application'] = (df['application_type'] == 'Joint App').astype(int)

print(f"Joint applications : {df['is_joint_application'].sum()}")
print(f"Individual apps    : {(df['is_joint_application'] == 0).sum()}")

# numeric joint columns - fill with 0 because individual borrowers
# simply have no co-borrower financial data, 0 is the correct representation
joint_numeric_cols = [
    'annual_inc_joint',       # no joint income if no co-borrower
    'dti_joint',              # no joint debt-to-income if no co-borrower
    'revol_bal_joint',        # no joint revolving balance if no co-borrower
    'sec_app_fico_range_low',         # no co-borrower FICO
    'sec_app_fico_range_high',        # no co-borrower FICO
    'sec_app_inq_last_6mths',         # no co-borrower inquiries
    'sec_app_mort_acc',               # no co-borrower mortgage accounts
    'sec_app_open_acc',               # no co-borrower open accounts
    'sec_app_revol_util',             # no co-borrower utilization
    'sec_app_open_act_il',            # no co-borrower installment loans
    'sec_app_num_rev_accts',          # no co-borrower revolving accounts
    'sec_app_chargeoff_within_12_mths',       # no co-borrower chargeoffs
    'sec_app_collections_12_mths_ex_med',     # no co-borrower collections
    'sec_app_mths_since_last_major_derog',    # no co-borrower derogatory marks
]
#inputation for numeric values
for col in joint_numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# categorical joint columns - fill with NOT_APPLICABLE
# because "unknown" would be wrong - we know exactly why it is missing
joint_categorical_cols = [
    'verification_status_joint',  # no verification if no co-borrower
]

for col in joint_categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('NOT_APPLICABLE')

# date joint columns - these cannot be filled with 0 or text
# we leave as NaT since we already know is_joint_application captures this
joint_date_cols = [
    'sec_app_earliest_cr_line',  # no co-borrower credit history date
]

for col in joint_date_cols:
    if col in df.columns:
        df[col] = df[col].fillna(pd.NaT)

# verify
print(f"\nMissing in annual_inc_joint after fill    : {df['annual_inc_joint'].isna().sum()}")
print(f"Missing in verification_status_joint after: {df['verification_status_joint'].isna().sum()}")

Joint applications : 27984
Individual apps    : 1338833

Missing in annual_inc_joint after fill    : 0
Missing in verification_status_joint after: 0


In [26]:
# Group 2 — Never Happened Columns (MAR with MNAR signal)
# these columns ask "how many months ago did bad thing X happen"
# if the bad thing never happened, the field is blank
# missing here carries a positive credit signal - never been delinquent
# we create a flag FIRST to preserve that signal, then fill with median
# median is calculated only from borrowers who DID experience the event


never_happened_cols = [
    'mths_since_last_delinq',           # never been delinquent
    'mths_since_last_record',           # never had a public record
    'mths_since_last_major_derog',      # never had a major derogatory mark
    'mths_since_recent_bc_dlq',         # never had a bank card delinquency
    'mths_since_recent_revol_delinq',   # never had a revolving delinquency
]

for col in never_happened_cols:
    if col in df.columns:
        # step 1: create binary flag FIRST to preserve the "never happened" signal
        # 1 means the event never happened (was missing)
        # 0 means it did happen at some point
        flag_name = col + '_never_happened'
        df[flag_name] = df[col].isna().astype(int)

        # step 2: fill NaN with median calculated only from borrowers
        # who actually experienced the event (non-null values)

        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

        print(f"{col}:")
        print(f"  flag '{flag_name}' - never happened: {df[flag_name].sum():,} borrowers")
        print(f"  filled NaN with median: {median_val:.1f}")
        print(f"  missing remaining: {df[col].isna().sum()}")
        print()

# verify all flags created
print("Flags created:")
never_flags = [c for c in df.columns if c.endswith('_never_happened')]
print(never_flags)

mths_since_last_delinq:
  flag 'mths_since_last_delinq_never_happened' - never happened: 688,927 borrowers
  filled NaN with median: 31.0
  missing remaining: 0

mths_since_last_record:
  flag 'mths_since_last_record_never_happened' - never happened: 1,134,225 borrowers
  filled NaN with median: 72.0
  missing remaining: 0

mths_since_last_major_derog:
  flag 'mths_since_last_major_derog_never_happened' - never happened: 1,006,511 borrowers
  filled NaN with median: 44.0
  missing remaining: 0

mths_since_recent_bc_dlq:
  flag 'mths_since_recent_bc_dlq_never_happened' - never happened: 1,042,362 borrowers
  filled NaN with median: 38.0
  missing remaining: 0

mths_since_recent_revol_delinq:
  flag 'mths_since_recent_revol_delinq_never_happened' - never happened: 909,229 borrowers
  filled NaN with median: 33.0
  missing remaining: 0

Flags created:
['mths_since_last_delinq_never_happened', 'mths_since_last_record_never_happened', 'mths_since_last_major_derog_never_happened', 'mths_sinc

In [27]:
#group 3: Old Loans Missing Credit Bureau Data (MCAR)
# these columns were not collected for older loans
# missing is purely due to when the loan was issued, not borrower behavior
# we create ONE flag to capture loan vintage before filling with median

group3_cols = [
    'open_acc_6m',        # credit lines opened in last 6 months
    'open_act_il',        # active installment loans
    'open_il_12m',        # installment loans opened in last 12 months
    'open_il_24m',        # installment loans opened in last 24 months
    'open_rv_12m',        # revolving accounts opened in last 12 months
    'open_rv_24m',        # revolving accounts opened in last 24 months
    'inq_last_12m',       # inquiries in last 12 months
    'inq_fi',             # finance company inquiries
    'total_bal_il',       # total installment loan balance
    'total_cu_tl',        # total credit union accounts
    'max_bal_bc',         # maximum bank card balance
    'all_util',           # overall utilization across all accounts
    'il_util',            # installment loan utilization
    'mths_since_rcnt_il', # months since most recent installment loan
]

# step 1: create a single flag capturing whether credit bureau data existed
# this captures the loan vintage signal without creating 14 separate flags
df['credit_bureau_data_available'] = (
    df['open_acc_6m'].notna()
).astype(int)

print(f"Loans WITH credit bureau data   : {df['credit_bureau_data_available'].sum()}")
print(f"Loans WITHOUT credit bureau data: {(df['credit_bureau_data_available'] == 0).sum()}")

# step 2: fill each column with its own median
# median is better than mean for financial data because it is not
# affected by extreme outliers
for col in group3_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {col} with median: {median_val:.2f}")

# verify
print(f"\nMissing in open_acc_6m after fill: {df['open_acc_6m'].isna().sum()}")
print(f"Missing in il_util after fill    : {df['il_util'].isna().sum()}")

Loans WITH credit bureau data   : 557518
Loans WITHOUT credit bureau data: 809299
Filled open_acc_6m with median: 1.00
Filled open_act_il with median: 2.00
Filled open_il_12m with median: 1.00
Filled open_il_24m with median: 1.00
Filled open_rv_12m with median: 1.00
Filled open_rv_24m with median: 2.00
Filled inq_last_12m with median: 2.00
Filled inq_fi with median: 1.00
Filled total_bal_il with median: 24051.00
Filled total_cu_tl with median: 0.00
Filled max_bal_bc with median: 4190.00
Filled all_util with median: 60.00
Filled il_util with median: 74.00
Filled mths_since_rcnt_il with median: 12.00

Missing in open_acc_6m after fill: 0
Missing in il_util after fill    : 0


In [28]:
#Group 4 — Moderate Random Gaps (Likely MCAR)
# these have small random gaps with no clear structural reason
# median for numeric columns, mode for emp_length

# emp_length: use mode because it represents categories like
# "2 years", "5 years" , the most common value is the best guess
emp_length_mode = df['emp_length'].mode()[0]
df['emp_length'] = df['emp_length'].fillna(emp_length_mode)
print(f"emp_length filled with mode: {emp_length_mode}")

# all other group 4 numeric columns: fill with median
group4_cols = [
    'mths_since_recent_inq',    # months since most recent inquiry
    'num_tl_120dpd_2m',         # accounts 120 days past due in last 2 months
    'mo_sin_old_il_acct',       # months since oldest installment account
    'pct_tl_nvr_dlq',           # percentage of accounts never delinquent
    'num_il_tl',                # total installment loan accounts
    'num_tl_op_past_12m',       # accounts opened in last 12 months
    'mo_sin_old_rev_tl_op',     # months since oldest revolving account
    'num_tl_30dpd',             # accounts 30 days past due
    'num_rev_tl_bal_gt_0',      # revolving accounts with balance over 0
    'num_rev_accts',            # total revolving accounts
    'num_bc_tl',                # total bank card accounts
    'tot_hi_cred_lim',          # total high credit limit
    'num_tl_90g_dpd_24m',       # accounts 90+ days past due in last 24 months
    'num_op_rev_tl',            # open revolving accounts
    'total_il_high_credit_limit', # total installment high credit limit
    'mo_sin_rcnt_rev_tl_op',    # months since most recent revolving account
    'mo_sin_rcnt_tl',           # months since most recent account
    'num_accts_ever_120_pd',    # accounts ever 120 days past due
    'total_rev_hi_lim',         # total revolving high credit limit
    'avg_cur_bal',              # average current balance
    'num_actv_bc_tl',           # active bank card accounts
    'tot_coll_amt',             # total collection amounts
    'tot_cur_bal',              # total current balance
    'num_actv_rev_tl',          # active revolving accounts
    'bc_util',                  # bank card utilization
    'percent_bc_gt_75',         # percentage of bank cards over 75% utilization
    'bc_open_to_buy',           # open to buy on bank cards
    'mths_since_recent_bc',     # months since most recent bank card
    'num_sats',                 # total satisfactory accounts
    'num_bc_sats',              # satisfactory bank card accounts
    'total_bc_limit',           # total bank card limit
    'total_bal_ex_mort',        # total balance excluding mortgage
    'mort_acc',                 # number of mortgage accounts
    'acc_open_past_24mths',     # accounts opened in past 24 months
]

for col in group4_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)


print(f"Missing in emp_length        : {df['emp_length'].isna().sum()}")
print(f"Missing in mort_acc          : {df['mort_acc'].isna().sum()}")
print(f"Missing in mths_since_recent_inq: {df['mths_since_recent_inq'].isna().sum()}")

emp_length filled with mode: 10.0
Missing in emp_length        : 0
Missing in mort_acc          : 0
Missing in mths_since_recent_inq: 0


In [29]:
#Group 5 — Tiny Gaps (MCAR)
# barely any missing values - simple median imputation
# the impact is negligible regardless of strategy chosen

group5_cols = [
    'revol_util',                   # credit utilization rate
    'dti',                          # debt to income ratio
    'pub_rec_bankruptcies',         # number of bankruptcy records
    'collections_12_mths_ex_med',   # collections in last 12 months
    'tax_liens',                    # number of tax liens
    'chargeoff_within_12_mths',     # charge offs in last 12 months
]

for col in group5_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {col} with median: {median_val:.4f}")

df['inq_last_6mths'] = df['inq_last_6mths'].fillna(df['inq_last_6mths'].median())

Filled revol_util with median: 52.1000
Filled dti with median: 17.6300
Filled pub_rec_bankruptcies with median: 0.0000
Filled collections_12_mths_ex_med with median: 0.0000
Filled tax_liens with median: 0.0000
Filled chargeoff_within_12_mths with median: 0.0000


In [30]:
#final verification
# check no missing values remain except sec_app_earliest_cr_line
# which we intentionally left as NaT
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("=== Remaining missing values ===")
if len(remaining_missing) == 0:
    print("No missing values remaining!")
else:
    print(remaining_missing.to_string())

print(f"\nFinal dataset shape: {df.shape}")
print(f"Total flag columns created: {len([c for c in df.columns if c.endswith('_flag') or c.endswith('_never_happened')])}")

=== Remaining missing values ===
member_id                   1366817
sec_app_earliest_cr_line    1346169
hardship_type               1359438
deferral_term               1359438
hardship_length             1359438

Final dataset shape: (1366817, 121)
Total flag columns created: 14


DUPLICATE

In [31]:
df.duplicated().sum()
# check for duplicate rows

np.int64(0)

DATA VISUALIZATION

FEATURE ENGINEERING

In [32]:


# 5. SUSPICIOUS OUTLIERS — CREATE INVESTIGATION FLAGS


# annual_inc: max is $10,999,200 which is almost $11M
# extremely unlikely for a personal loan platform
# flag incomes above $500k as suspicious outliers for investigation
df['annual_inc_flag'] = (df['annual_inc'] > 500000).astype(int)

# revol_bal: max is $2,904,836 which is nearly $3M revolving balance
# flag balances above $500k as suspicious
df['revol_bal_flag'] = (df['revol_bal'] > 500000).astype(int)

# tot_coll_amt: max is $9,152,545 which is extremely high for collection amount
# flag amounts above $100k as suspicious
df['tot_coll_amt_flag'] = (df['tot_coll_amt'] > 100000).astype(int)

# mths_since_rcnt_il: max is 511 months (42+ years) which is suspicious
# flag values above 300 months (25 years) as suspicious
df['mths_since_rcnt_il_flag'] = (df['mths_since_rcnt_il'] > 300).astype(int)

# mths_since_recent_bc: max is 639 months (53+ years) which is suspicious
# flag values above 300 months as suspicious
df['mths_since_recent_bc_flag'] = (df['mths_since_recent_bc'] > 300).astype(int)

# mo_sin_rcnt_rev_tl_op: max is 438 months (36+ years) which is suspicious
# flag values above 300 months as suspicious
df['mo_sin_rcnt_rev_tl_op_flag'] = (df['mo_sin_rcnt_rev_tl_op'] > 300).astype(int)

# mo_sin_rcnt_tl: max is 314 months (26+ years) which is suspicious
# flag values above 300 months as suspicious
df['mo_sin_rcnt_tl_flag'] = (df['mo_sin_rcnt_tl'] > 300).astype(int)

# earliest_cr_line: some dates go back to 1934 which means the borrower
# would be 80+ years old, likely a data entry error
# flag credit lines opened before 1950 as suspicious
df['earliest_cr_line_flag'] = (df['earliest_cr_line'].dt.year < 1950).astype(int)

# delinq_amnt: max is $249,925 with a mean of only $14.92
# extreme outlier, flag amounts above $10,000 for investigation
df['delinq_amnt_flag'] = (df['delinq_amnt'] > 10000).astype(int)

/tmp/ipykernel_407/1884244948.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['annual_inc_flag'] = (df['annual_inc'] > 500000).astype(int)
/tmp/ipykernel_407/1884244948.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['revol_bal_flag'] = (df['revol_bal'] > 500000).astype(int)
/tmp/ipykernel_407/1884244948.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

In [33]:
#Currently you have fico_range_low and fico_range_high as two separate columns. They always differ by exactly 4 points so keeping both is redundant.

#FICO Score Features
# fico_avg: single clean credit score instead of two near-identical columns
# the midpoint is the most honest representation of the borrower's score
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

# fico_risk_bucket: group scores into standard credit risk categories
# this captures non-linear relationships that a model might miss
# (difference between 580 and 620 is much more meaningful than 750 vs 790)
def fico_bucket(score):
    if score >= 800:
        return 'exceptional'
    elif score >= 740:
        return 'very_good'
    elif score >= 670:
        return 'good'
    elif score >= 580:
        return 'fair'
    else:
        return 'poor'

df['fico_risk_bucket'] = df['fico_avg'].apply(fico_bucket)

# drop the original two columns since fico_avg replaces them
df = df.drop(columns=['fico_range_low', 'fico_range_high'])

print("FICO features created")
print(df['fico_risk_bucket'].value_counts())
print(df['fico_avg'].describe())

/tmp/ipykernel_407/1489571922.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
/tmp/ipykernel_407/1489571922.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['fico_risk_bucket'] = df['fico_avg'].apply(fico_bucket)


FICO features created
fico_risk_bucket
good           977699
fair           242080
very_good      131436
exceptional     15602
Name: count, dtype: int64
count    1.366817e+06
mean     6.981246e+02
std      3.181650e+01
min      6.270000e+02
25%      6.720000e+02
50%      6.920000e+02
75%      7.120000e+02
max      8.475000e+02
Name: fico_avg, dtype: float64


In [34]:
#Ratio Features
#These combine two existing columns into a single more meaningful ratio.
#Why ratios are powerful: A person borrowing $10,000 with a $200,000 income is very different from a person borrowing $10,000 with a $15,000 income. The raw loan amount alone does not tell you that — but the ratio does.
# loan_to_income: what percentage of annual income is being borrowed
# high ratio = borrower is taking on a lot relative to their earnings
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

# revol_bal_to_income: revolving debt relative to annual income
# shows how much existing debt burden the borrower is carrying
df['revol_bal_to_income'] = df['revol_bal'] / df['annual_inc']

# total_debt_to_income: combines dti concept with actual balance
# more complete picture of debt burden than dti alone
df['total_debt_to_income'] = df['tot_cur_bal'] / (df['annual_inc'] + 1)
# +1 avoids division by zero for the rare annual_inc = 0 cases

# open_acc_ratio: proportion of total accounts that are still open
# shows how active the borrower's credit profile is
df['open_acc_ratio'] = df['open_acc'] / (df['total_acc'] + 1)

# delinq_to_acc_ratio: proportion of accounts ever delinquent
# directly measures historical repayment problems
df['delinq_to_acc_ratio'] = df['num_accts_ever_120_pd'] / (df['total_acc'] + 1)

print("Ratio features created successfully")
print(df[['loan_to_income', 'revol_bal_to_income',
          'total_debt_to_income', 'open_acc_ratio',
          'delinq_to_acc_ratio']].describe())

Ratio features created successfully


/tmp/ipykernel_407/3682234839.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']
/tmp/ipykernel_407/3682234839.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['revol_bal_to_income'] = df['revol_bal'] / df['annual_inc']
/tmp/ipykernel_407/3682234839.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) ins

       loan_to_income  revol_bal_to_income  total_debt_to_income  \
count    1.366817e+06         1.366808e+06          1.366817e+06   
mean              inf                  inf          4.047537e+01   
std               NaN                  NaN          3.608938e+03   
min      1.714286e-04         0.000000e+00          0.000000e+00   
25%      1.250000e-01         9.813333e-02          5.360616e-01   
50%      2.000000e-01         1.792203e-01          1.253579e+00   
75%      2.916667e-01         2.952966e-01          2.760836e+00   
max               inf                  inf          1.339445e+06   

       open_acc_ratio  delinq_to_acc_ratio  
count    1.366817e+06         1.366817e+06  
mean     4.761497e-01         1.853308e-02  
std      1.600288e-01         4.694366e-02  
min      0.000000e+00         0.000000e+00  
25%      3.571429e-01         0.000000e+00  
50%      4.615385e-01         0.000000e+00  
75%      5.833333e-01         0.000000e+00  
max      1.555556e+00      

In [35]:
df['credit_age'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365

/tmp/ipykernel_407/3171009478.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['credit_age'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365


FEATURE SELECTION

In [36]:
df = df.drop(columns=['sec_app_earliest_cr_line'])

In [37]:
df['credit_age'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365

In [38]:
X = df.drop(columns=['issue_d', 'earliest_cr_line'])

In [39]:
#Drop pure identifier
df = df.drop(columns=['id'])
#Drop constant columns
constant_cols = df.nunique()[df.nunique() <= 1].index.tolist()
df = df.drop(columns=constant_cols)
print(constant_cols)

['member_id', 'policy_code', 'hardship_type', 'deferral_term', 'hardship_length']


In [40]:
# Target variable
y = df['loan_status']

# Features (drop the target column)
X = df.drop(columns=['loan_status'])

In [41]:
# Columns where almost everyone has the same value tell the model nothing.
from sklearn.feature_selection import VarianceThreshold

# select only numeric columns for variance check
numeric_cols = df.select_dtypes(include=['float64', 'int64',
                                          'Int64']).columns.tolist()

# remove target variable from this check
numeric_cols = [c for c in numeric_cols if c != 'target']

# calculate variance for each column
variances = df[numeric_cols].var().sort_values()

# show columns with very low variance
low_variance = variances[variances < 0.01]
print(f"Columns with near-zero variance: {len(low_variance)}")
print(low_variance)

Columns with near-zero variance: 22
mo_sin_old_il_acct_flag               0.000001
mo_sin_rcnt_tl_flag                   0.000001
total_rev_hi_lim_flag                 0.000002
earliest_cr_line_flag                 0.000007
tot_hi_cred_lim_flag                  0.000009
mo_sin_rcnt_rev_tl_op_flag            0.000012
tot_coll_amt_flag                     0.000035
mths_since_rcnt_il_flag               0.000052
revol_bal_flag                        0.000143
dti_flag                               0.00016
sec_app_revol_util_flag               0.000247
delinq_amnt_flag                       0.00033
mths_since_recent_bc_flag             0.000627
num_tl_120dpd_2m                      0.000839
annual_inc_flag                       0.001303
delinq_to_acc_ratio                   0.002204
sec_app_collections_12_mths_ex_med    0.003003
sec_app_chargeoff_within_12_mths      0.003256
revol_util_flag                       0.003477
num_tl_30dpd                          0.003686
all_util_flag           

In [42]:
#High Correlation Between Features
import seaborn as sns
import matplotlib.pyplot as plt

# calculate correlation matrix for numeric columns
corr_matrix = df[numeric_cols].corr().abs()

# find pairs with correlation above 0.90
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = [
    (col, row, upper_triangle.loc[row, col])
    for col in upper_triangle.columns
    for row in upper_triangle.index
    if upper_triangle.loc[row, col] > 0.90
]

print(f"Highly correlated pairs (>0.90): {len(high_corr_pairs)}")
for col1, col2, corr in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True):
    print(f"  {col1} vs {col2}: {corr:.4f}")

Highly correlated pairs (>0.90): 7
  sec_app_fico_range_high vs sec_app_fico_range_low: 1.0000
  num_sats vs open_acc: 0.9841
  tot_hi_cred_lim vs tot_cur_bal: 0.9833
  num_rev_tl_bal_gt_0 vs num_actv_rev_tl: 0.9820
  is_joint_application vs dti_joint: 0.9246
  sec_app_num_rev_accts vs sec_app_open_acc: 0.9199
  delinq_to_acc_ratio vs num_accts_ever_120_pd: 0.9083


In [43]:
#Correlation With Target
# calculate correlation of each feature with target
target_corr = df[numeric_cols].corrwith(df['loan_status']).abs().sort_values(ascending=False)

print("Top 20 features most correlated with target:")
print(target_corr.head(20))

print("\nBottom 20 features least correlated with target:")
print(target_corr.tail(20))

# features with near-zero correlation with target
weak_features = target_corr[target_corr < 0.01]
print(f"\nFeatures with < 0.01 correlation with target: {len(weak_features)}")
print(weak_features)

Top 20 features most correlated with target:
loan_status                     1.000000
term                            0.181314
fico_avg                        0.129553
dti                             0.106282
acc_open_past_24mths            0.099621
num_tl_op_past_12m              0.084205
credit_bureau_data_available    0.079019
mort_acc                        0.076271
bc_open_to_buy                  0.075796
tot_hi_cred_lim                 0.074590
avg_cur_bal                     0.074043
open_rv_24m                     0.073589
loan_amnt                       0.071719
num_actv_rev_tl                 0.069219
total_bc_limit                  0.068151
num_rev_tl_bal_gt_0             0.067464
tot_cur_bal                     0.066430
open_acc_ratio                  0.062897
inq_last_6mths                  0.062164
percent_bc_gt_75                0.060647
dtype: float64

Bottom 20 features least correlated with target:
delinq_amnt                   0.002752
num_tl_30dpd                  0

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2767: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]


In [44]:
#Drop based on filter results
# collect columns to drop based on filter results
cols_to_drop = []

# 1. near zero variance columns
cols_to_drop.extend(low_variance.index.tolist())

# 2. from high correlation pairs - drop the second column in each pair
# since the first one captures the same information
high_corr_to_drop = list(set([col2 for col1, col2, corr in high_corr_pairs]))
cols_to_drop.extend(high_corr_to_drop)

# 3. weak correlation with target
cols_to_drop.extend(weak_features.index.tolist())



print(f"Total columns to drop from filter methods: {len(cols_to_drop)}")
print(cols_to_drop)

# drop them
df = df.drop(columns=cols_to_drop)
print(f"\nDataset shape after filter selection: {df.shape}")

Total columns to drop from filter methods: 62
['mo_sin_old_il_acct_flag', 'mo_sin_rcnt_tl_flag', 'total_rev_hi_lim_flag', 'earliest_cr_line_flag', 'tot_hi_cred_lim_flag', 'mo_sin_rcnt_rev_tl_op_flag', 'tot_coll_amt_flag', 'mths_since_rcnt_il_flag', 'revol_bal_flag', 'dti_flag', 'sec_app_revol_util_flag', 'delinq_amnt_flag', 'mths_since_recent_bc_flag', 'num_tl_120dpd_2m', 'annual_inc_flag', 'delinq_to_acc_ratio', 'sec_app_collections_12_mths_ex_med', 'sec_app_chargeoff_within_12_mths', 'revol_util_flag', 'num_tl_30dpd', 'all_util_flag', 'acc_now_delinq', 'num_accts_ever_120_pd', 'tot_cur_bal', 'sec_app_fico_range_low', 'open_acc', 'num_actv_rev_tl', 'sec_app_open_acc', 'dti_joint', 'revol_util_flag', 'num_rev_accts', 'mths_since_last_delinq', 'num_il_tl', 'annual_inc_flag', 'mths_since_recent_revol_delinq', 'mths_since_recent_bc_dlq', 'sec_app_mort_acc', 'mths_since_rcnt_il', 'mths_since_last_major_derog', 'chargeoff_within_12_mths', 'acc_now_delinq', 'emp_length', 'revol_bal_flag', 't

MODELING AND EVALUATION

In [45]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)


In [ ]:
# Convert categorical variables to numeric
X = pd.get_dummies(X, drop_first=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
X_train = X_train.drop(columns=['issue_d', 'earliest_cr_line'])
X_test = X_test.drop(columns=['issue_d', 'earliest_cr_line'])

import numpy as np

# Replace inf/-inf with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)


In [ ]:
# Scale
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

In [50]:
import numpy as np

np.isinf(X_train).sum().sum()

np.int64(582)

.
If skewness is above 2, the distribution has an extreme tail (like annual_inc where a few millionaires sit far from everyone else). Capping alone won't fix the shape, so you log transform to compress the whole scale down.


If skewness is between 0.5 and 2, the shape is mostly fine but the tail is a bit too long. You just cap the extreme values at the 99th percentile (winsorize) and leave the rest alone.


If skewness is below 0.5, the distribution is roughly symmetric, meaning outliers are genuine anomalies that clearly don't belong — not part of a natural tail. IQR works perfectly here because it measures spread around the center and catches those isolated freaks cleanly.


The key rule to remember: never use IQR on skewed data, because on a lopsided distribution its fence lands inside perfectly valid data and starts flagging normal observations as outliers.

In [ ]:
#outliers handling

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────
# STEP 1: Identify ALL numeric columns from training set
# ─────────────────────────────────────────────────────────
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Total numeric columns: {len(numeric_cols)}")

# ─────────────────────────────────────────────────────────
# STEP 2: Compute skewness on train — let skewness decide treatment
# ─────────────────────────────────────────────────────────
skewness = X_train[numeric_cols].skew().sort_values(ascending=False)

# Categorize by skewness level
highly_skewed  = skewness[skewness.abs() > 2].index.tolist()    # log transform
moderately_skewed = skewness[(skewness.abs() > 0.5) &
                              (skewness.abs() <= 2)].index.tolist()  # winsorize
roughly_normal = skewness[skewness.abs() <= 0.5].index.tolist()  # IQR or leave

print(f"\nHighly skewed    (|skew| > 2):   {len(highly_skewed)} cols → Log transform")
print(f"Moderately skewed (0.5–2):        {len(moderately_skewed)} cols → Winsorize")
print(f"Roughly normal   (|skew| ≤ 0.5): {len(roughly_normal)} cols → IQR / leave")
print(f"\nHighly skewed columns:\n{highly_skewed}")


# --- Percentile caps (for moderately skewed) ---
pct_caps = {}
for col in moderately_skewed:
    pct_caps[col] = {
        'lower': X_train[col].quantile(0.01),
        'upper': X_train[col].quantile(0.99)
    }

# --- IQR bounds (for roughly normal) ---
iqr_bounds = {}
for col in roughly_normal:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    iqr_bounds[col] = {
        'lower': Q1 - 3 * IQR,
        'upper': Q3 + 3 * IQR
    }

# --- Log shift (for highly skewed, handles zeros/negatives) ---
log_shifts = {}
for col in highly_skewed:
    min_val = X_train[col].min()
    log_shifts[col] = abs(min_val) + 1 if min_val <= 0 else 0



# --- Winsorize moderately skewed ---
for col in moderately_skewed:
    lower = pct_caps[col]['lower']
    upper = pct_caps[col]['upper']
    X_train[col] = X_train[col].clip(lower=lower, upper=upper)
    X_test[col]  = X_test[col].clip(lower=lower, upper=upper)

# --- Log transform highly skewed ---
for col in highly_skewed:
    shift = log_shifts[col]
    X_train[col] = np.log1p(X_train[col] + shift)
    X_test[col]  = np.log1p(X_test[col]  + shift)

# --- Flag + clip IQR outliers for roughly normal ---
for col in roughly_normal:
    lower = iqr_bounds[col]['lower']
    upper = iqr_bounds[col]['upper']
    # Flag first, then clip
    X_train[col + '_outlier_flag'] = ((X_train[col] < lower) | (X_train[col] > upper)).astype(int)
    X_test[col  + '_outlier_flag'] = ((X_test[col]  < lower) | (X_test[col]  > upper)).astype(int)
    X_train[col] = X_train[col].clip(lower=lower, upper=upper)
    X_test[col]  = X_test[col].clip(lower=lower, upper=upper)


skewness_after = X_train[numeric_cols].skew().sort_values(ascending=False)

comparison = pd.DataFrame({
    'skew_before': skewness[numeric_cols],
    'skew_after' : skewness_after[numeric_cols]
})
comparison['improved'] = comparison['skew_after'].abs() < comparison['skew_before'].abs()

print("\nSkewness before vs after (top 15 most skewed):")
print(comparison.sort_values('skew_before', ascending=False).head(15).to_string())
print(f"\nColumns improved: {comparison['improved'].sum()} / {len(numeric_cols)}")


In [ ]:
#Logistic regression
print("\n" + "="*55)
print("MODEL 1: Logistic Regression (Baseline)")
print("="*55)

lr = LogisticRegression(max_iter=500, random_state=42, class_weight="balanced")
lr.fit(X_train_s, y_train)

y_pred_lr  = lr.predict(X_test_s)
y_prob_lr  = lr.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred_lr, target_names=["Bad","Good"]))

In [ ]:



 #Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)          # trees don't need scaling

y_pred_rf  = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=["Bad","Good"]))

In [ ]:
def metrics(y_true, y_pred, y_prob, name):
    return {
        "Model"     : name,
        "Accuracy"  : round(accuracy_score(y_true, y_pred),4),
        "Precision" : round(precision_score(y_true, y_pred),4),
        "Recall"    : round(recall_score(y_true, y_pred),4),
        "F1-Score"  : round(f1_score(y_true, y_pred),4),
        "ROC-AUC"   : round(roc_auc_score(y_true, y_prob),4),
    }

results = pd.DataFrame([
    metrics(y_test, y_pred_lr, y_prob_lr, "Logistic Regression"),
    metrics(y_test, y_pred_rf, y_prob_rf, "Random Forest"),
])
print("\n── Summary Metrics ──")
print(results.to_string(index=False))

# Cross-validation (5-fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = cross_val_score(lr, X_train_s, y_train, cv=cv, scoring="roc_auc")
rf_cv = cross_val_score(rf, X_train,   y_train, cv=cv, scoring="roc_auc")
print(f"\nCV ROC-AUC  LR: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}")
print(f"CV ROC-AUC  RF: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

In [ ]:
PALETTE = {"Good":"#2ecc71", "Bad":"#e74c3c",
           "LR":"#3498db",   "RF":"#9b59b6"}

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor("#0f1117")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

axis_kw = dict(facecolor="#1a1d27", labelcolor="#cfd3dc",
               titlecolor="white",  tick_params=dict(colors="#cfd3dc"))

def style_ax(ax, title):
    ax.set_facecolor("#1a1d27")
    ax.set_title(title, color="white", fontsize=11, fontweight="bold", pad=8)
    ax.tick_params(colors="#cfd3dc")
    for spine in ax.spines.values():
        spine.set_edgecolor("#2c2f3e")
    ax.xaxis.label.set_color("#cfd3dc")
    ax.yaxis.label.set_color("#cfd3dc")


# 7a. Confusion Matrices
for col_idx, (y_pred, name) in enumerate(
        [(y_pred_lr,"Logistic Regression"), (y_pred_rf,"Random Forest")]):
    ax = fig.add_subplot(gs[0, col_idx])
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Bad","Good"], yticklabels=["Bad","Good"],
                ax=ax, linewidths=0.5, linecolor="#2c2f3e",
                annot_kws={"size":13, "color":"white"})
    style_ax(ax, f"Confusion Matrix – {name}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")


#  ROC Curves
ax_roc = fig.add_subplot(gs[0, 2])
for y_prob, name, color in [
        (y_prob_lr, "Logistic Regression", PALETTE["LR"]),
        (y_prob_rf, "Random Forest",       PALETTE["RF"])]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax_roc.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")
ax_roc.plot([0,1],[0,1],"--", color="#555", lw=1)
style_ax(ax_roc, "ROC Curves")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(fontsize=8, facecolor="#1a1d27", labelcolor="white")


#  Metrics Bar Chart
ax_bar = fig.add_subplot(gs[1, :2])
metrics_cols = ["Accuracy","Precision","Recall","F1-Score","ROC-AUC"]
x = np.arange(len(metrics_cols))
w = 0.35
bars_lr = ax_bar.bar(x - w/2, results[results.Model=="Logistic Regression"][metrics_cols].values[0],
                     width=w, color=PALETTE["LR"], label="Logistic Regression", alpha=0.9)
bars_rf = ax_bar.bar(x + w/2, results[results.Model=="Random Forest"][metrics_cols].values[0],
                     width=w, color=PALETTE["RF"], label="Random Forest", alpha=0.9)
ax_bar.set_xticks(x); ax_bar.set_xticklabels(metrics_cols)
ax_bar.set_ylim(0, 1.12)
ax_bar.set_ylabel("Score")
for bars in [bars_lr, bars_rf]:
    for b in bars:
        ax_bar.text(b.get_x()+b.get_width()/2, b.get_height()+0.012,
                    f"{b.get_height():.3f}", ha="center", va="bottom",
                    color="white", fontsize=7.5)
style_ax(ax_bar, "Model Comparison – Evaluation Metrics")
ax_bar.legend(fontsize=9, facecolor="#1a1d27", labelcolor="white")


#  CV ROC-AUC Box
ax_cv = fig.add_subplot(gs[1, 2])
bp = ax_cv.boxplot(
    [lr_cv, rf_cv], labels=["LR","RF"],
    patch_artist=True, widths=0.4,
    boxprops=dict(linewidth=1.5),
    medianprops=dict(color="white", linewidth=2),
    whiskerprops=dict(color="#cfd3dc"),
    capprops=dict(color="#cfd3dc"),
    flierprops=dict(marker="o", color="#e74c3c", markersize=5)
)
for patch, color in zip(bp["boxes"], [PALETTE["LR"], PALETTE["RF"]]):
    patch.set_facecolor(color); patch.set_alpha(0.8)
style_ax(ax_cv, "5-Fold CV ROC-AUC")
ax_cv.set_ylabel("ROC-AUC")
ax_cv.set_ylim(0.4, 1.0)


#. Feature Importances (RF)
ax_fi = fig.add_subplot(gs[2, :])
importances = pd.Series(rf.feature_importances_, index=X.columns)
top15 = importances.nlargest(15).sort_values()
colors = [PALETTE["RF"] if v > top15.mean() else "#7f8c8d" for v in top15]
top15.plot(kind="barh", ax=ax_fi, color=colors, edgecolor="none")
style_ax(ax_fi, "Top 15 Feature Importances (Random Forest)")
ax_fi.set_xlabel("Importance Score")


# Main title
fig.suptitle("Lending Club – Borrower Creditworthiness Modeling",
             fontsize=15, fontweight="bold", color="white", y=0.98)

